# 01 — Contrat de données (Parquet) & IO


Objectifs : définir un **schéma stable**, valider, convertir CSV→Parquet, et garantir l'intégrité (NaN/Inf, monotonicité).


In [ ]:

import pandas as pd, numpy as np
from pathlib import Path
REQUIRED = ["sample_id","wavenumber_cm1","intensity"]
def validate_schema(df: pd.DataFrame, min_points=128):
    miss = [c for c in REQUIRED if c not in df.columns]
    if miss: raise ValueError(f"Colonnes manquantes: {miss}")
    if not np.issubdtype(df["wavenumber_cm1"].dtype, np.number): raise TypeError("wavenumber_cm1 doit être numérique")
    if not np.issubdtype(df["intensity"].dtype, np.number): raise TypeError("intensity doit être numérique")
    if len(df)<min_points: raise ValueError("Trop peu de points")
    x = df["wavenumber_cm1"].to_numpy()
    y = df["intensity"].to_numpy()
    if not (x[1:]>=x[:-1]).all(): raise ValueError("wavenumber_cm1 non croissant")
    if not (np.isfinite(x).all() and np.isfinite(y).all()): raise ValueError("NaN/Inf détectés")
    return True

# Démo dataset synthétique
def demo_spectrum(n=2048, start=100., stop=3500., seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(start, stop, n)
    y = np.sin(x/80.0) * 0.02 + 0.05*rng.standard_normal(n)
    y += np.exp(-0.5*((x-520)/6)**2) * (1.0+0.1*rng.normal(size=n))
    df = pd.DataFrame({"sample_id":["demo"]*n, "wavenumber_cm1":x, "intensity":y})
    return df

df = demo_spectrum()
assert validate_schema(df)
df.head()


In [ ]:

# Conversion CSV -> Parquet (fallback local); essaie la fonction projet si dispo
import sys
try:
    from optech_raman_ai.data.convert_csv_to_parquet import convert_folder as convert_csv_folder
    HAVE = True
except Exception as e:
    HAVE = False
    print("[WARN] Module convert_csv_folder indisponible:", e)

from pathlib import Path
raw = Path("data_raw"); raw.mkdir(exist_ok=True)
(df).to_csv(raw/"example.csv", index=False)

out = Path("data_processed"); out.mkdir(exist_ok=True)
if HAVE:
    convert_csv_folder(raw, out)
else:
    import pandas as pd
    for f in raw.glob("*.csv"):
        d = pd.read_csv(f)
        validate_schema(d, min_points=10)
        d.to_parquet(out/(f.stem+".parquet"), index=False)
sorted(out.glob("*.parquet"))
